In [ ]:
"""

Student: Emad Fattah 
Instructor: Randall Granier
Class: MSSE 640 
Week 7: Vibe Coding Mini-Assignment #3. State-Transitions/Control Flow Testing




ATM Testing Demo — Python
=========================
Demonstrates three testing disciplines applied to a simple ATM state machine:

  1. State Transition Testing  — every valid and invalid state change is exercised
  2. Control Flow Testing      — every branch / conditional path is covered
  3. Data Flow Testing         — variables are traced from definition → use → kill

Sunny Day scenarios: the happy path (things work as expected)
Rainy Day scenarios: error paths, boundary conditions, and guard failures

Run with:
    python atm_testing_demo.py
"""

import unittest
from datetime import datetime
from typing import Optional

# ─────────────────────────────────────────────────────────────────────────────
# DOMAIN MODEL
# ─────────────────────────────────────────────────────────────────────────────

class AtmState:
    IDLE = "IDLE"
    AWAITING_PIN = "AWAITING_PIN"
    AUTHENTICATED = "AUTHENTICATED"
    CARD_BLOCKED = "CARD_BLOCKED"


class Account:
    """Represents a bank account linked to a card."""

    def __init__(self, card_number: str, pin: str, holder_name: str, balance: float):
        if not card_number or not isinstance(card_number, str):
            raise ValueError("card_number must be a non-empty string")
        if not pin or not isinstance(pin, str):
            raise ValueError("pin must be a non-empty string")
        if not holder_name or not isinstance(holder_name, str):
            raise ValueError("holder_name must be a non-empty string")
        if not isinstance(balance, (int, float)) or balance < 0:
            raise ValueError("balance must be a non-negative number")

        self.card_number: str = card_number
        self.pin: str = pin
        self.holder_name: str = holder_name
        self.balance: float = round(float(balance), 2)
        self.is_blocked: bool = False


class ATM:
    """
    ATM state machine.

    States:
        IDLE           → insert a valid card  → AWAITING_PIN
        AWAITING_PIN   → correct PIN          → AUTHENTICATED
        AWAITING_PIN   → 3 wrong PINs         → CARD_BLOCKED
        AUTHENTICATED  → eject card           → IDLE
        CARD_BLOCKED   → eject card           → IDLE
    """

    MAX_PIN_ATTEMPTS = 3
    MAX_WITHDRAWAL = 10_000.0

    def __init__(self, atm_cash: float = 10_000.0):
        if not isinstance(atm_cash, (int, float)) or atm_cash < 0:
            raise ValueError("atm_cash must be a non-negative number")

        self._accounts: dict[str, Account] = {}
        self._state: str = AtmState.IDLE
        self._atm_cash: float = round(float(atm_cash), 2)
        self._active_card: Optional[str] = None
        self._pin_attempts: int = 0
        self.last_receipt: Optional[str] = None

    # ── Registration ──────────────────────────────────────────────────────────

    def register_account(self, account: Account) -> None:
        """Add an account to this ATM's known cards."""
        if not isinstance(account, Account):
            raise TypeError("account must be an Account instance")
        self._accounts[account.card_number] = account

    # ── Read-only properties ──────────────────────────────────────────────────

    @property
    def state(self) -> str:
        return self._state

    @property
    def atm_cash(self) -> float:
        return self._atm_cash

    @property
    def pin_attempts(self) -> int:
        return self._pin_attempts

    @property
    def active_account(self) -> Optional[Account]:
        if self._active_card is None:
            return None
        return self._accounts.get(self._active_card)

    # ── Actions ───────────────────────────────────────────────────────────────

    def insert_card(self, card_number: str) -> str:
        """
        Insert a card. Transitions IDLE → AWAITING_PIN (or CARD_BLOCKED).

        Returns a status message string.
        Raises ValueError on bad input; RuntimeError on invalid state transition.
        """
        if not card_number or not isinstance(card_number, str):
            raise ValueError("card_number must be a non-empty string")

        card_number = card_number.strip()

        if self._state != AtmState.IDLE:
            raise RuntimeError(
                f"Cannot insert card: machine is {self._state}. Eject current card first."
            )

        account = self._accounts.get(card_number)
        if account is None:
            # Card not recognised — machine stays IDLE
            return f"Card {card_number!r} not recognised."

        if account.is_blocked:
            self._state = AtmState.CARD_BLOCKED
            self._active_card = card_number
            return f"Card {card_number!r} is permanently blocked. Please contact your bank."

        self._state = AtmState.AWAITING_PIN
        self._active_card = card_number
        self._pin_attempts = 0
        return f"Card accepted. Please enter your PIN ({self.MAX_PIN_ATTEMPTS} attempts allowed)."

    def enter_pin(self, pin: str) -> str:
        """
        Validate a PIN. Transitions AWAITING_PIN → AUTHENTICATED or CARD_BLOCKED.

        Returns a status message string.
        Raises ValueError on bad input; RuntimeError on invalid state transition.
        """
        if pin is None or not isinstance(pin, str):
            raise ValueError("pin must be a string")

        if self._state != AtmState.AWAITING_PIN:
            raise RuntimeError(
                f"Cannot enter PIN: machine is {self._state}."
            )

        account = self.active_account
        if account is None:
            # Defensive guard — should not happen in normal flow
            self._state = AtmState.IDLE
            self._active_card = None
            self._pin_attempts = 0
            raise RuntimeError("Session corrupt: no active account found. Card ejected.")

        if account.pin == pin:
            self._state = AtmState.AUTHENTICATED
            self._pin_attempts = 0
            return f"PIN accepted. Welcome, {account.holder_name}."

        # Wrong PIN
        self._pin_attempts += 1
        remaining = self.MAX_PIN_ATTEMPTS - self._pin_attempts

        if remaining <= 0:
            account.is_blocked = True
            self._state = AtmState.CARD_BLOCKED
            return (
                f"Incorrect PIN. Card blocked after {self.MAX_PIN_ATTEMPTS} failed attempts."
            )

        return f"Incorrect PIN. {remaining} attempt(s) remaining."

    def check_balance(self) -> float:
        """
        Return the authenticated account's balance.

        Raises RuntimeError if not AUTHENTICATED.
        """
        if self._state != AtmState.AUTHENTICATED:
            raise RuntimeError(
                f"Cannot check balance: machine is {self._state}."
            )
        # active_account is guaranteed non-None when AUTHENTICATED
        return self.active_account.balance  # type: ignore[union-attr]

    def withdraw(self, amount: float) -> str:
        """
        Withdraw cash.

        Checks:
            1. Machine must be AUTHENTICATED
            2. Amount must be a positive, finite number
            3. Amount must not exceed single-transaction cap
            4. ATM must have enough cash
            5. Account must have enough balance

        On success: deducts from account and ATM; generates a receipt.
        Returns a status message string.
        """
        if self._state != AtmState.AUTHENTICATED:
            raise RuntimeError(
                f"Cannot withdraw: machine is {self._state}."
            )

        # ── Data-flow: definition of `amount` as a validated float ────────────
        try:
            amount = float(amount)
        except (TypeError, ValueError):
            return "Invalid amount: must be a number."

        if amount <= 0 or not (amount == amount):   # NaN check via reflexivity
            return "Invalid amount: must be a positive number."

        if amount > self.MAX_WITHDRAWAL:
            return (
                f"Withdrawal exceeds single-transaction limit of "
                f"${self.MAX_WITHDRAWAL:,.2f}."
            )

        account = self.active_account  # type: ignore[union-attr]

        if self._atm_cash <= 0:
            return "ATM is out of cash. Please visit another location."

        if amount > self._atm_cash:
            return (
                f"ATM only has ${self._atm_cash:,.2f} available. "
                f"Please request a smaller amount."
            )

        if amount > account.balance:
            return (
                f"Insufficient funds. Account balance is ${account.balance:,.2f}."
            )

        # ── Data-flow: `amount` used to mutate balance and atm_cash ──────────
        account.balance = round(account.balance - amount, 2)
        self._atm_cash = round(self._atm_cash - amount, 2)

        # ── Data-flow: `receipt` defined here, stored for later assertion ──────
        now = datetime.now()
        receipt = (
            "================================\n"
            "       TRANSACTION RECEIPT      \n"
            "================================\n"
            f"Date:    {now.strftime('%Y-%m-%d')}\n"
            f"Time:    {now.strftime('%H:%M:%S')}\n"
            f"Card:    **** **** **** {account.card_number[-4:]}\n"
            f"Holder:  {account.holder_name}\n"
            "--------------------------------\n"
            f"Drawn:   ${amount:>10,.2f}\n"
            f"Balance: ${account.balance:>10,.2f}\n"
            "================================\n"
        )
        self.last_receipt = receipt  # data-flow: kill of previous receipt value

        return f"Dispensing ${amount:,.2f}. Take your cash and receipt."

    def eject_card(self) -> str:
        """
        Eject card and return to IDLE from any state except IDLE.

        Raises RuntimeError if already IDLE.
        """
        if self._state == AtmState.IDLE:
            raise RuntimeError("No card inserted.")

        self._state = AtmState.IDLE
        self._active_card = None
        self._pin_attempts = 0
        return "Card ejected. Have a great day."


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS used across tests
# ─────────────────────────────────────────────────────────────────────────────

def make_atm_with_accounts(atm_cash: float = 10_000.0) -> ATM:
    """Return a fresh ATM loaded with four test accounts."""
    atm = ATM(atm_cash=atm_cash)
    atm.register_account(Account("1111", "1234", "Emad Fattah", 2500.00))
    atm.register_account(Account("2222", "5678", "Bob Martinez", 840.50))
    atm.register_account(Account("3333", "9999", "Carol Chen", 12_750.00))
    atm.register_account(Account("4444", "0000", "David Kim", 50.00))
    return atm


# ─────────────────────────────────────────────────────────────────────────────
# TEST SUITES
# ─────────────────────────────────────────────────────────────────────────────

class TestStateTransitions(unittest.TestCase):
    """
    ╔══════════════════════════════════════════════════════════════╗
    ║  1. STATE TRANSITION TESTING                                  ║
    ║                                                              ║
    ║  Verifies every edge in the state diagram:                   ║
    ║                                                              ║
    ║  IDLE ──insert valid──► AWAITING_PIN                         ║
    ║  IDLE ──insert blocked─► CARD_BLOCKED                        ║
    ║  AWAITING_PIN ──correct PIN──► AUTHENTICATED                 ║
    ║  AWAITING_PIN ──3 wrong PINs──► CARD_BLOCKED                 ║
    ║  AUTHENTICATED ──eject──► IDLE                               ║
    ║  CARD_BLOCKED ──eject──► IDLE                                ║
    ║                                                              ║
    ║  Also verifies INVALID transitions raise RuntimeError.        ║
    ╚══════════════════════════════════════════════════════════════╝
    """

    def setUp(self):
        self.atm = make_atm_with_accounts()

    # ── Sunny Day ─────────────────────────────────────────────────────────────

    def test_ST_SD_01_idle_to_awaiting_pin(self):
        """SUNNY: Insert a valid card → machine moves to AWAITING_PIN."""
        msg = self.atm.insert_card("1111")
        self.assertEqual(self.atm.state, AtmState.AWAITING_PIN)
        self.assertIn("PIN", msg)

    def test_ST_SD_02_awaiting_pin_to_authenticated(self):
        """SUNNY: Correct PIN → machine moves to AUTHENTICATED."""
        self.atm.insert_card("1111")
        msg = self.atm.enter_pin("1234")
        self.assertEqual(self.atm.state, AtmState.AUTHENTICATED)
        self.assertIn("Welcome", msg)

    def test_ST_SD_03_authenticated_to_idle(self):
        """SUNNY: Eject after authentication → machine returns to IDLE."""
        self.atm.insert_card("1111")
        self.atm.enter_pin("1234")
        msg = self.atm.eject_card()
        self.assertEqual(self.atm.state, AtmState.IDLE)
        self.assertIn("ejected", msg)

    def test_ST_SD_04_card_blocked_to_idle(self):
        """SUNNY: Eject a blocked card → machine returns to IDLE."""
        self.atm.insert_card("1111")
        for _ in range(3):
            self.atm.enter_pin("wrong")
        self.assertEqual(self.atm.state, AtmState.CARD_BLOCKED)
        self.atm.eject_card()
        self.assertEqual(self.atm.state, AtmState.IDLE)

    def test_ST_SD_05_re_insert_after_session(self):
        """SUNNY: Full session completes; a second card can be inserted."""
        self.atm.insert_card("1111")
        self.atm.enter_pin("1234")
        self.atm.eject_card()
        # Now insert a different card
        self.atm.insert_card("2222")
        self.assertEqual(self.atm.state, AtmState.AWAITING_PIN)

    # ── Rainy Day ─────────────────────────────────────────────────────────────

    def test_ST_RD_01_insert_card_when_not_idle(self):
        """RAINY: Inserting a card when machine is not IDLE raises RuntimeError."""
        self.atm.insert_card("1111")  # now AWAITING_PIN
        with self.assertRaises(RuntimeError):
            self.atm.insert_card("2222")

    def test_ST_RD_02_enter_pin_when_idle(self):
        """RAINY: Entering PIN when IDLE raises RuntimeError."""
        with self.assertRaises(RuntimeError):
            self.atm.enter_pin("1234")

    def test_ST_RD_03_enter_pin_when_authenticated(self):
        """RAINY: Entering PIN when AUTHENTICATED raises RuntimeError."""
        self.atm.insert_card("1111")
        self.atm.enter_pin("1234")
        with self.assertRaises(RuntimeError):
            self.atm.enter_pin("1234")

    def test_ST_RD_04_eject_when_idle(self):
        """RAINY: Ejecting when no card inserted raises RuntimeError."""
        with self.assertRaises(RuntimeError):
            self.atm.eject_card()

    def test_ST_RD_05_insert_pre_blocked_card(self):
        """RAINY: Inserting a card that is already blocked → CARD_BLOCKED immediately."""
        # Pre-block the account externally
        self.atm._accounts["1111"].is_blocked = True
        self.atm.insert_card("1111")
        self.assertEqual(self.atm.state, AtmState.CARD_BLOCKED)

    def test_ST_RD_06_withdraw_when_not_authenticated(self):
        """RAINY: Withdrawal when machine is IDLE raises RuntimeError."""
        with self.assertRaises(RuntimeError):
            self.atm.withdraw(100)

    def test_ST_RD_07_check_balance_when_not_authenticated(self):
        """RAINY: check_balance when machine is IDLE raises RuntimeError."""
        with self.assertRaises(RuntimeError):
            self.atm.check_balance()


class TestControlFlow(unittest.TestCase):
    """
    ╔══════════════════════════════════════════════════════════════╗
    ║  2. CONTROL FLOW TESTING                                     ║
    ║                                                              ║
    ║  Exercises every branch of every conditional in the ATM.     ║
    ║                                                              ║
    ║  Branches covered:                                           ║
    ║    insert_card:  valid card / unknown card / already blocked ║
    ║    enter_pin:    correct / wrong (partial) / wrong (final)   ║
    ║    withdraw:     success / non-numeric / negative / zero /   ║
    ║                  over-limit / ATM empty / ATM low /          ║
    ║                  account empty / account insufficient        ║
    ╚══════════════════════════════════════════════════════════════╝
    """

    def setUp(self):
        self.atm = make_atm_with_accounts()

    def _authenticate(self, card="1111", pin="1234"):
        self.atm.insert_card(card)
        self.atm.enter_pin(pin)

    # ── Sunny Day ─────────────────────────────────────────────────────────────

    def test_CF_SD_01_withdraw_exact_balance(self):
        """SUNNY: Withdraw entire account balance — both checks pass."""
        self._authenticate()
        msg = self.atm.withdraw(2500.00)
        self.assertIn("Dispensing", msg)
        self.assertEqual(self.atm.active_account.balance, 0.0)

    def test_CF_SD_02_withdraw_partial_balance(self):
        """SUNNY: Partial withdrawal — account still has funds remaining."""
        self._authenticate()
        self.atm.withdraw(100.00)
        self.assertAlmostEqual(self.atm.active_account.balance, 2400.00, places=2)

    def test_CF_SD_03_check_balance_branch(self):
        """SUNNY: check_balance returns correct value (exercises the success branch)."""
        self._authenticate()
        balance = self.atm.check_balance()
        self.assertEqual(balance, 2500.00)

    def test_CF_SD_04_one_wrong_pin_then_correct(self):
        """SUNNY: One wrong PIN (partial-fail branch) then correct PIN."""
        self.atm.insert_card("1111")
        msg_wrong = self.atm.enter_pin("0000")
        self.assertIn("2 attempt(s) remaining", msg_wrong)
        self.assertEqual(self.atm.state, AtmState.AWAITING_PIN)
        msg_ok = self.atm.enter_pin("1234")
        self.assertEqual(self.atm.state, AtmState.AUTHENTICATED)
        self.assertIn("Welcome", msg_ok)

    def test_CF_SD_05_two_wrong_then_correct(self):
        """SUNNY: Two wrong PINs (1 remaining branch) then correct PIN."""
        self.atm.insert_card("1111")
        self.atm.enter_pin("bad1")
        msg = self.atm.enter_pin("bad2")
        self.assertIn("1 attempt(s) remaining", msg)
        self.atm.enter_pin("1234")
        self.assertEqual(self.atm.state, AtmState.AUTHENTICATED)

    # ── Rainy Day ─────────────────────────────────────────────────────────────

    def test_CF_RD_01_unknown_card(self):
        """RAINY: insert_card with unrecognised card — machine stays IDLE."""
        msg = self.atm.insert_card("9999")
        self.assertIn("not recognised", msg)
        self.assertEqual(self.atm.state, AtmState.IDLE)

    def test_CF_RD_02_three_wrong_pins_blocks_card(self):
        """RAINY: Three consecutive wrong PINs → CARD_BLOCKED."""
        self.atm.insert_card("1111")
        for _ in range(2):
            self.atm.enter_pin("bad")
        msg = self.atm.enter_pin("bad")
        self.assertEqual(self.atm.state, AtmState.CARD_BLOCKED)
        self.assertIn("blocked", msg)
        self.assertTrue(self.atm._accounts["1111"].is_blocked)

    def test_CF_RD_03_withdraw_non_numeric_string(self):
        """RAINY: Non-numeric withdrawal → invalid-amount branch."""
        self._authenticate()
        msg = self.atm.withdraw("abc")
        self.assertIn("Invalid amount", msg)

    def test_CF_RD_04_withdraw_negative_amount(self):
        """RAINY: Negative amount → guard branch fires."""
        self._authenticate()
        msg = self.atm.withdraw(-50)
        self.assertIn("Invalid amount", msg)

    def test_CF_RD_05_withdraw_zero(self):
        """RAINY: Zero amount → guard branch fires."""
        self._authenticate()
        msg = self.atm.withdraw(0)
        self.assertIn("Invalid amount", msg)

    def test_CF_RD_06_withdraw_over_limit(self):
        """RAINY: Amount exceeds single-transaction cap ($10,000)."""
        self._authenticate("3333", "9999")   # Carol has $12,750
        msg = self.atm.withdraw(10_001)
        self.assertIn("limit", msg)

    def test_CF_RD_07_withdraw_more_than_atm_cash(self):
        """RAINY: ATM has less cash than requested — ATM-cash guard fires."""
        atm = ATM(atm_cash=200.0)
        atm.register_account(Account("5555", "1111", "Eve Adams", 5000.0))
        atm.insert_card("5555")
        atm.enter_pin("1111")
        msg = atm.withdraw(500)
        self.assertIn("ATM only has", msg)

    def test_CF_RD_08_atm_completely_empty(self):
        """RAINY: ATM has zero cash — out-of-cash guard fires."""
        atm = ATM(atm_cash=0.0)
        atm.register_account(Account("5555", "1111", "Eve Adams", 5000.0))
        atm.insert_card("5555")
        atm.enter_pin("1111")
        msg = atm.withdraw(100)
        self.assertIn("out of cash", msg)

    def test_CF_RD_09_withdraw_more_than_account_balance(self):
        """RAINY: Requested amount exceeds account balance — balance guard fires."""
        self._authenticate("4444", "0000")   # David Kim, $50
        msg = self.atm.withdraw(200)
        self.assertIn("Insufficient funds", msg)

    def test_CF_RD_10_withdraw_non_numeric_none(self):
        """RAINY: None passed as amount — handled gracefully (no crash)."""
        self._authenticate()
        msg = self.atm.withdraw(None)
        self.assertIn("Invalid amount", msg)


class TestDataFlow(unittest.TestCase):
    """
    ╔══════════════════════════════════════════════════════════════╗
    ║  3. DATA FLOW TESTING                                        ║
    ║                                                              ║
    ║  Traces key variables through define → use → kill paths.     ║
    ║                                                              ║
    ║  Variables traced:                                           ║
    ║    account.balance   — defined on Account; mutated by        ║
    ║                        withdraw; read by check_balance       ║
    ║    atm._atm_cash     — defined on ATM init; mutated by       ║
    ║                        withdraw; checked before dispensing   ║
    ║    atm.last_receipt  — undefined (None) initially; defined   ║
    ║                        by withdraw success; re-defined by    ║
    ║                        a second withdraw (old value killed)  ║
    ║    pin_attempts      — reset to 0 on insert; incremented     ║
    ║                        per wrong PIN; reset to 0 on success  ║
    ╚══════════════════════════════════════════════════════════════╝
    """

    def setUp(self):
        self.atm = make_atm_with_accounts()

    def _authenticate(self, card="1111", pin="1234"):
        self.atm.insert_card(card)
        self.atm.enter_pin(pin)

    # ── Sunny Day ─────────────────────────────────────────────────────────────

    def test_DF_SD_01_balance_defined_and_read(self):
        """SUNNY: account.balance is defined on creation; readable via check_balance."""
        # Define → use
        self._authenticate()
        balance = self.atm.check_balance()  # use
        self.assertEqual(balance, 2500.00)

    def test_DF_SD_02_balance_defined_mutated_used(self):
        """SUNNY: account.balance define → withdraw mutates it → check_balance reads updated value."""
        self._authenticate()
        initial = self.atm.check_balance()    # use (pre-mutation)
        self.atm.withdraw(400.00)             # mutation (kill of old value, define of new)
        updated = self.atm.check_balance()    # use (post-mutation)
        self.assertAlmostEqual(updated, initial - 400.00, places=2)

    def test_DF_SD_03_atm_cash_decreases_after_withdrawal(self):
        """SUNNY: atm_cash defined at init; used and killed by withdraw."""
        initial_cash = self.atm.atm_cash     # use: read before mutation
        self._authenticate()
        self.atm.withdraw(300.00)            # mutation: kill old, define new
        self.assertAlmostEqual(self.atm.atm_cash, initial_cash - 300.00, places=2)

    def test_DF_SD_04_receipt_defined_by_withdrawal(self):
        """SUNNY: last_receipt is None before first withdrawal; defined after it."""
        self.assertIsNone(self.atm.last_receipt)          # pre-condition (None)
        self._authenticate()
        self.atm.withdraw(100.00)                         # define receipt
        self.assertIsNotNone(self.atm.last_receipt)       # use: receipt now exists
        self.assertIn("TRANSACTION RECEIPT", self.atm.last_receipt)

    def test_DF_SD_05_receipt_redefined_on_second_withdrawal(self):
        """SUNNY: Second withdrawal kills the old receipt and defines a new one."""
        self._authenticate()
        self.atm.withdraw(100.00)
        first_receipt = self.atm.last_receipt    # capture old value

        self.atm.withdraw(50.00)                 # kill first, define second
        second_receipt = self.atm.last_receipt   # use new value

        self.assertNotEqual(first_receipt, second_receipt)
        self.assertIn("50.00", second_receipt)

    def test_DF_SD_06_pin_attempts_reset_on_insert(self):
        """SUNNY: pin_attempts defined as 0 at insert_card (fresh session)."""
        self.atm.insert_card("1111")
        self.assertEqual(self.atm.pin_attempts, 0)  # use: should be 0

    def test_DF_SD_07_pin_attempts_increment_then_reset_on_success(self):
        """SUNNY: pin_attempts increments on wrong PIN; killed (reset to 0) on correct PIN."""
        self.atm.insert_card("1111")
        self.atm.enter_pin("bad")          # define: attempts → 1
        self.assertEqual(self.atm.pin_attempts, 1)
        self.atm.enter_pin("1234")         # kill: attempts reset to 0
        self.assertEqual(self.atm.pin_attempts, 0)

    def test_DF_SD_08_multiple_withdrawals_accumulate_deductions(self):
        """SUNNY: Sequential withdrawals compound the balance mutation correctly."""
        self._authenticate()
        self.atm.withdraw(200)   # balance: 2500 → 2300
        self.atm.withdraw(300)   # balance: 2300 → 2000
        self.atm.withdraw(500)   # balance: 2000 → 1500
        self.assertAlmostEqual(self.atm.check_balance(), 1500.00, places=2)
        self.assertAlmostEqual(self.atm.atm_cash, 10_000 - 1000, places=2)

    # ── Rainy Day ─────────────────────────────────────────────────────────────

    def test_DF_RD_01_balance_unchanged_after_failed_withdrawal(self):
        """RAINY: A failed withdrawal must NOT mutate account.balance."""
        self._authenticate("4444", "0000")   # David Kim, $50
        before = self.atm.check_balance()
        self.atm.withdraw(500)               # should fail — insufficient funds
        after = self.atm.check_balance()
        self.assertEqual(before, after)

    def test_DF_RD_02_atm_cash_unchanged_after_failed_withdrawal(self):
        """RAINY: A failed withdrawal must NOT mutate atm_cash."""
        self._authenticate()
        cash_before = self.atm.atm_cash
        self.atm.withdraw(-99)               # invalid amount guard fires
        self.assertEqual(self.atm.atm_cash, cash_before)

    def test_DF_RD_03_receipt_remains_none_after_failed_withdrawal(self):
        """RAINY: A failed withdrawal must NOT overwrite last_receipt."""
        self._authenticate("4444", "0000")
        self.assertIsNone(self.atm.last_receipt)
        self.atm.withdraw(5000)              # fails — insufficient funds
        self.assertIsNone(self.atm.last_receipt)

    def test_DF_RD_04_receipt_preserved_after_second_withdrawal_fails(self):
        """RAINY: Good receipt followed by a failed withdrawal keeps the good receipt."""
        self._authenticate()
        self.atm.withdraw(100)
        good_receipt = self.atm.last_receipt
        self.atm.withdraw(99_999)            # fails — over limit
        self.assertEqual(self.atm.last_receipt, good_receipt)

    def test_DF_RD_05_pin_attempts_accumulate_to_block(self):
        """RAINY: pin_attempts increments correctly across all 3 failed attempts."""
        self.atm.insert_card("1111")
        self.assertEqual(self.atm.pin_attempts, 0)

        self.atm.enter_pin("x")
        self.assertEqual(self.atm.pin_attempts, 1)

        self.atm.enter_pin("x")
        self.assertEqual(self.atm.pin_attempts, 2)

        self.atm.enter_pin("x")   # 3rd wrong → card blocked
        # After blocking, state is CARD_BLOCKED; attempts stay at 3 (not reset)
        self.assertEqual(self.atm.pin_attempts, 3)
        self.assertEqual(self.atm.state, AtmState.CARD_BLOCKED)

    def test_DF_RD_06_account_created_with_invalid_balance_raises(self):
        """RAINY: Account.__init__ rejects negative balance — data validated at definition."""
        with self.assertRaises(ValueError):
            Account("X", "0000", "Hacker", -100)

    def test_DF_RD_07_atm_created_with_invalid_cash_raises(self):
        """RAINY: ATM.__init__ rejects negative cash — data validated at definition."""
        with self.assertRaises(ValueError):
            ATM(atm_cash=-500)

    def test_DF_RD_08_insert_card_empty_string_raises(self):
        """RAINY: insert_card with empty string raises ValueError before state change."""
        with self.assertRaises(ValueError):
            self.atm.insert_card("")
        self.assertEqual(self.atm.state, AtmState.IDLE)

    def test_DF_RD_09_enter_pin_none_raises(self):
        """RAINY: enter_pin(None) raises ValueError — input validated at entry point."""
        self.atm.insert_card("1111")
        with self.assertRaises(ValueError):
            self.atm.enter_pin(None)


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("=" * 70)
    print("  ATM TESTING DEMO — State Transitions / Control Flow / Data Flow")
    print("=" * 70)
    print()

    # Run all three suites and collect results
    loader = unittest.TestLoader()
    suite = unittest.TestSuite()

    for cls in [TestStateTransitions, TestControlFlow, TestDataFlow]:
        suite.addTests(loader.loadTestsFromTestCase(cls))

    runner = unittest.TextTestRunner(verbosity=2)
    result = runner.run(suite)

    print()
    print("=" * 70)
    total = result.testsRun
    passed = total - len(result.failures) - len(result.errors)
    print(f"  RESULTS: {passed}/{total} tests passed")
    if result.failures or result.errors:
        print("  FAILURES / ERRORS — see output above")
    else:
        print("  ALL TESTS PASSED")
    print("=" * 70)


# ── Interactive ATM — it asks YOU for the card and PIN ──

atm = make_atm_with_accounts()
print("Welcome to the ATM!")
print("-" * 40)

# Step 1: ask for the card number
card = input("Please insert your card (enter card number): ").strip()
print(atm.insert_card(card))

# Step 2: ask for the PIN (up to 3 tries)
while atm.state == AtmState.AWAITING_PIN:
    pin = input("Please enter your PIN: ").strip()
    print(atm.enter_pin(pin))

# Step 3: if PIN was correct, ask what to withdraw
if atm.state == AtmState.AUTHENTICATED:
    print(f"Your balance is: ${atm.check_balance():,.2f}")
    amount = float(input("How much would you like to withdraw? $"))
    print(atm.withdraw(amount))
    if atm.last_receipt:
        print()
        print(atm.last_receipt)
    print(atm.eject_card())
elif atm.state == AtmState.CARD_BLOCKED:
    print("Sorry, your card has been blocked.")
    print(atm.eject_card())
else:
    print("Session ended — card was not accepted.")




test_ST_RD_01_insert_card_when_not_idle (__main__.TestStateTransitions.test_ST_RD_01_insert_card_when_not_idle)
RAINY: Inserting a card when machine is not IDLE raises RuntimeError. ... ok
test_ST_RD_02_enter_pin_when_idle (__main__.TestStateTransitions.test_ST_RD_02_enter_pin_when_idle)
RAINY: Entering PIN when IDLE raises RuntimeError. ... ok
test_ST_RD_03_enter_pin_when_authenticated (__main__.TestStateTransitions.test_ST_RD_03_enter_pin_when_authenticated)
RAINY: Entering PIN when AUTHENTICATED raises RuntimeError. ... ok
test_ST_RD_04_eject_when_idle (__main__.TestStateTransitions.test_ST_RD_04_eject_when_idle)
RAINY: Ejecting when no card inserted raises RuntimeError. ... ok
test_ST_RD_05_insert_pre_blocked_card (__main__.TestStateTransitions.test_ST_RD_05_insert_pre_blocked_card)
RAINY: Inserting a card that is already blocked → CARD_BLOCKED immediately. ... ok
test_ST_RD_06_withdraw_when_not_authenticated (__main__.TestStateTransitions.test_ST_RD_06_withdraw_when_not_authentica

  ATM TESTING DEMO — State Transitions / Control Flow / Data Flow


  RESULTS: 44/44 tests passed
  ALL TESTS PASSED
Welcome to the ATM!
----------------------------------------


Please insert your card (enter card number):  1111


Card accepted. Please enter your PIN (3 attempts allowed).


Please enter your PIN:  1234


PIN accepted. Welcome, Emad Fattah.
Your balance is: $2,500.00
